# LexIA — Phase 6 : Monitoring Langfuse

Ce notebook retrace la mise en place du monitoring de LexIA avec Langfuse.
Il couvre :
- Pourquoi monitorer un RAG en production
- Architecture des traces (spans retrieval + generation)
- Test de connexion et création de traces
- Intégration dans la chain RAG
- Ce qu'on observe dans le dashboard Langfuse

**Stack** : Langfuse v4.5, cloud.langfuse.com (gratuit)

## 1. Pourquoi monitorer un RAG ?

Un RAG en production peut dégrader silencieusement sans qu'on s'en aperçoive :
- Le corpus vieillit (lois modifiées non mises à jour)
- Les questions évoluent (nouveaux sujets hors périmètre)
- La latence augmente (index qui grossit)
- Le LLM hallucine sur certains patterns de questions

### Langfuse vs alternatives

| Outil | Open source | Self-hostable | Gratuit |
|---|---|---|---|
| **Langfuse** | ✅ | ✅ | ✅ |
| LangSmith | ❌ | ❌ | Limité |
| Helicone | ❌ | ❌ | Limité |
| Arize | ❌ | ❌ | Limité |

Langfuse est préféré car open source et déployable on-premise — compatible RGPD
pour les entreprises françaises qui ne peuvent pas envoyer leurs données aux US.

### Architecture des traces LexIA

```
Trace : lexia-rag-query
├── Span : retrieval
│   ├── input  : question, filter_code
│   └── output : n_chunks, articles, avg_score
└── Span : generation (type=generation)
    ├── input  : question
    ├── model  : llama-3.3-70b-versatile
    └── output : réponse juridique
```

## 2. Imports et configuration

In [ ]:
import sys
import os
import time
import json
from dotenv import load_dotenv
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

sys.path.insert(0, '/workspaces/lexia')
load_dotenv('/workspaces/lexia/.env')

print('Imports OK')
print(f'LANGFUSE_PUBLIC_KEY : {os.getenv("LANGFUSE_PUBLIC_KEY", "NON CONFIGURÉ")[:12]}...')

## 3. Test de connexion Langfuse

In [ ]:
from monitoring.langfuse_client import test_connection, langfuse

ok = test_connection()
if ok:
    print('✓ Langfuse connecté — les traces seront visibles sur cloud.langfuse.com')
else:
    print('✗ Vérifiez LANGFUSE_PUBLIC_KEY et LANGFUSE_SECRET_KEY dans .env')

## 4. Création d'une trace manuelle

In [ ]:
from monitoring.langfuse_client import trace_rag_query, score_trace

# Trace manuelle avec des données fictives
trace_id = trace_rag_query(
    question="Mon employeur peut-il me licencier pendant un arrêt maladie ?",
    answer="Selon l'article L1226-9 du Code du travail, au cours des périodes de suspension...",
    contexts=[
        {
            "article_num":     "L1226-9",
            "code_name":       "Code du travail",
            "relevance_score": 0.821,
            "page_content":    "Au cours des périodes de suspension du contrat de travail...",
        },
        {
            "article_num":     "L1226-11",
            "code_name":       "Code du travail",
            "relevance_score": 0.759,
            "page_content":    "Lorsque, à l'issue d'un délai d'un mois...",
        },
    ],
    filter_code="Code du travail",
    latency_ms=1234,
    user_id="notebook-test",
)

if trace_id:
    print(f'✓ Trace créée : {trace_id}')
    url = f'https://cloud.langfuse.com/trace/{trace_id}'
    print(f'  Voir : {url}')

In [ ]:
# Ajoute un score de feedback
if trace_id:
    score_trace(
        trace_id=trace_id,
        score=1.0,
        comment="Réponse correcte — article L1226-9 bien cité"
    )
    print('✓ Score de feedback ajouté (1.0 = positif)')

## 5. Trace via une vraie requête RAG

In [ ]:
from rag.chain import ask

# La trace Langfuse est automatique dans ask()
start = time.time()
response = ask(
    question="Quel est le délai de rétractation pour un achat en ligne ?",
    filter_code="Code de la consommation",
    stream=False,
)
latency = (time.time() - start) * 1000

print(f'Latence : {latency:.0f}ms')
print(f'Réponse : {response[:300]}...')
print()
print('✓ Trace automatiquement envoyée à Langfuse')
print('  Vérifiez sur cloud.langfuse.com')

## 6. Simulation de plusieurs traces — dashboard

In [ ]:
# Génère 5 traces avec des questions variées
# pour peupler le dashboard Langfuse

test_questions = [
    ("Mon employeur peut-il me licencier pendant un arrêt maladie ?", "Code du travail"),
    ("Quelles sont les conditions pour une rupture conventionnelle ?", "Code du travail"),
    ("Quel est le délai de rétractation pour un achat en ligne ?", "Code de la consommation"),
    ("Qu'est-ce que le harcèlement moral au travail ?", "Code du travail"),
    ("Quel est le temps qu'il fait en Bretagne ?", None),  # hors sujet
]

trace_ids = []
latencies = []

for i, (question, filter_code) in enumerate(test_questions):
    print(f'[{i+1}/5] {question[:50]}...')
    start = time.time()
    response = ask(question, filter_code=filter_code, stream=False)
    latency = (time.time() - start) * 1000
    latencies.append(latency)
    print(f'  Latence : {latency:.0f}ms')

print(f'\n✓ {len(test_questions)} traces envoyées à Langfuse')
print(f'  Latence moyenne : {sum(latencies)/len(latencies):.0f}ms')
print(f'  Latence P95     : {sorted(latencies)[int(len(latencies)*0.95)]:.0f}ms')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

questions_labels = [q[:35] + '...' for q, _ in test_questions]
colors = ['#1D9E75' if f else '#D85A30' for _, f in test_questions]

bars = ax.barh(questions_labels, latencies, color=colors, edgecolor='white')
for bar, lat in zip(bars, latencies):
    ax.text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2,
            f'{lat:.0f}ms', va='center', fontsize=9)

ax.set_xlabel('Latence (ms)')
ax.set_title('Latence par requête LexIA')
ax.set_xlim(0, max(latencies) * 1.2)
ax.grid(axis='x', alpha=0.3)

patch_ok  = mpatches.Patch(color='#1D9E75', label='Question dans le périmètre')
patch_hors = mpatches.Patch(color='#D85A30', label='Question hors périmètre')
ax.legend(handles=[patch_ok, patch_hors], fontsize=9)

plt.tight_layout()
plt.savefig('/workspaces/lexia/notebooks/monitoring_latency.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Ce qu'on observe dans le dashboard Langfuse

### Métriques disponibles

| Métrique | Utilité |
|---|---|
| **Latence P50/P95** | Détecter les dégradations de performance |
| **Taux de réponses vides** | Questions hors périmètre = corpus à enrichir |
| **avg_relevance_score** | Drift du retrieval dans le temps |
| **Scores feedback** | Satisfaction utilisateur réelle |
| **Questions fréquentes** | Orienter l'enrichissement du corpus |

### Alertes à configurer en production

```python
# Exemple d'alerte : avg_score < 0.6 sur 7 jours
# → possible drift de qualité du retrieval
# → vérifier si le corpus est à jour

# Exemple d'alerte : taux réponses vides > 20%
# → beaucoup de questions hors périmètre
# → enrichir le corpus ou élargir le périmètre
```

## 8. Conclusions

### Ce qui est en place
- ✅ Trace automatique à chaque requête via `ask()`
- ✅ Spans retrieval + generation séparés
- ✅ Score feedback utilisateur via `score_trace()`
- ✅ Graceful degradation — Langfuse non bloquant

### Évolutions production
- Self-hosting Langfuse (Docker) pour la souveraineté des données
- Alertes automatiques sur drift de qualité
- Dashboard métriques business (questions par jour, taux de satisfaction)
- Intégration RAGAS post-hoc sur les traces pour évaluation continue